In [1]:
import json
import pandas as pd
import time
import datetime
import orjson

In [2]:
start_time_full = time.time()
start_time_full_str = datetime.datetime.now().strftime("%H:%M:%S")

In [3]:
with open('../raw_data/events.json', 'r') as f:
    data = json.load(f)

In [4]:
start_time = time.time()
start_time_str = datetime.datetime.now().strftime("%H:%M:%S")

In [5]:
data_df = pd.json_normalize(data,sep='_')

# New loop free

In [41]:
import pandas as pd
import time

def normalize_params(df, column_name, prefix):
    print(f"\n=== Processing column: {column_name} with prefix: '{prefix}' ===")
    total_start = time.time()

    # ✅ Correct .copy() usage
    df = df[[column_name]].copy()

    # Step 1: Explode
    start = time.time()
    print("[1/6] Exploding list column...")
    exploded = df.explode(column_name).reset_index(names='original_index')
    print(f"    → Exploded rows: {len(exploded)}")
    print(f"    ⏱️ Step time: {time.time() - start:.3f}s\n")

    # Step 2: Normalize JSON structure
    start = time.time()
    print("[2/6] Normalizing JSON data...")
    params_df = pd.json_normalize(exploded[column_name])
    print(f"    → Normalized columns: {list(params_df.columns)}")
    print(f"    ⏱️ Step time: {time.time() - start:.3f}s\n")

    # Step 3: Extract key/value pairs
    start = time.time()
    print("[3/6] Extracting key/value pairs...")
    exploded['key'] = params_df['key']
    exploded['value'] = (
        params_df['value.string_value']
        .fillna(params_df['value.int_value'])
        .fillna(params_df['value.float_value'])
        .fillna(params_df['value.double_value'])
    )
    print("    → Sample keys:", exploded['key'].dropna().unique()[:5])
    print(f"    ⏱️ Step time: {time.time() - start:.3f}s\n")

    # Step 4: Pivot key/value to wide format
    start = time.time()
    print("[4/6] Pivoting key-value pairs to wide format...")
    normalized_df = (
        exploded
        .pivot(index='original_index', columns='key', values='value')
        .reset_index()
        .sort_index(axis=1)
    )
    print(f"    → Normalized shape: {normalized_df.shape}")
    print(f"    ⏱️ Step time: {time.time() - start:.3f}s\n")

    # Step 5: Add prefix
    start = time.time()
    print("[5/6] Adding prefix to columns...")
    normalized_df = normalized_df.add_prefix(prefix)
    normalized_df = normalized_df.rename(columns={f"{prefix}original_index": "original_index"})
    print(f"    → Prefixed columns: {list(normalized_df.columns)[:5]}...")
    print(f"    ⏱️ Step time: {time.time() - start:.3f}s\n")

    # Step 6: Merge back with original DataFrame
    start = time.time()
    print("[6/6] Merging normalized columns back to original DataFrame...")
    merged = (
        df.reset_index(names='original_index')
        .merge(normalized_df, on='original_index', how='left')
        .drop(columns=['original_index', column_name])
    )
    print(f"    → Final shape after merge: {merged.shape}")
    print(f"    ⏱️ Step time: {time.time() - start:.3f}s\n")

    total_time = time.time() - total_start
    print(f"=== ✅ Done in {total_time:.3f} seconds ===\n")

    return merged


In [43]:
df_1 = normalize_params(data_df,'event_params','ep_')
df_1


=== Processing column: event_params with prefix: 'ep_' ===
[1/6] Exploding list column...
    → Exploded rows: 2987091
    ⏱️ Step time: 1.173s

[2/6] Normalizing JSON data...
    → Normalized columns: ['key', 'value.string_value', 'value.int_value', 'value.float_value', 'value.double_value']
    ⏱️ Step time: 20.959s

[3/6] Extracting key/value pairs...
    → Sample keys: ['batch_page_id' 'source' 'page_location' 'engaged_session_event'
 'campaign']
    ⏱️ Step time: 2.329s

[4/6] Pivoting key-value pairs to wide format...
    → Normalized shape: (260864, 29)
    ⏱️ Step time: 2.728s

[5/6] Adding prefix to columns...
    → Prefixed columns: ['ep_batch_ordering_id', 'ep_batch_page_id', 'ep_campaign', 'ep_click_classes', 'ep_click_id']...
    ⏱️ Step time: 0.399s

[6/6] Merging normalized columns back to original DataFrame...
    → Final shape after merge: (260864, 28)
    ⏱️ Step time: 0.667s

=== ✅ Done in 28.306 seconds ===



,ep_batch_ordering_id,ep_batch_page_id,ep_campaign,ep_click_classes,ep_click_id,ep_click_text,ep_click_url,ep_content,ep_engaged_session_event,ep_engagement_time_msec,...,ep_link_url,ep_medium,ep_page_location,ep_page_referrer,ep_page_title,ep_percent_scrolled,ep_session_engaged,ep_source,ep_term,ep_value
0,1.0,1731513969906.0,productpickup,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,4pc,https://www.shopko.com/eye-care/il/crystal-lak...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,prm,NaN,NaN
1,1.0,1731513969906.0,productpickup,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,4pc,https://www.shopko.com/eye-care/il/crystal-lak...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,1.0,prm,NaN,NaN
2,1.0,1731513969906.0,productpickup,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,4pc,https://www.shopko.com/eye-care/il/crystal-lak...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,prm,NaN,NaN
3,2.0,1731513969906.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,158.0,...,NaN,NaN,https://www.shopko.com/eye-care/il/crystal-lak...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,NaN,NaN,NaN
4,2.0,1731513969906.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1120.0,...,NaN,NaN,https://www.shopko.com/eye-care/il/crystal-lak...,NaN,Eye Care Center in Crystal Lake - Shopko Optical,10.0,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260859,2.0,1731552185721.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,31.0,...,NaN,NaN,https://www.shopko.com/eye-care/?utm_source=em...,NaN,Eye Exam Center Locations Near You,NaN,0,NaN,NaN,NaN
260860,2.0,1731552185721.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,3255.0,...,NaN,NaN,https://www.shopko.com/eye-care/?utm_source=em...,NaN,Eye Exam Center Locations Near You,10.0,0,NaN,NaN,NaN
260861,2.0,1731552185721.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0,...,NaN,NaN,https://www.shopko.com/eye-care/?utm_source=em...,NaN,Eye Exam Center Locations Near You,20.0,0,NaN,NaN,NaN
260862,2.0,1731552185721.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,...,NaN,NaN,https://www.shopko.com/eye-care/?utm_source=em...,NaN,Eye Exam Center Locations Near You,30.0,0,NaN,NaN,NaN


In [44]:
df_1 = normalize_params(df_1,'user_properties','up_')
df_1


=== Processing column: user_properties with prefix: 'up_' ===


KeyError: "None of [Index(['user_properties'], dtype='object')] are in the [columns]"

# Old

In [9]:
import pandas as pd
import time

def extract_nested_params(df, column_name, prefix):
    print(f"\n=== Processing column: '{column_name}' with prefix: '{prefix}' ===")
    total_start = time.time()

    # Step 1: Start extraction
    start = time.time()
    print("[1/4] Extracting key/value pairs from nested structure...")
    data = []
    for items_list in df[column_name]:
        row_dict = {
            item["key"]: next(
                v
                for k, v in item["value"].items()
                if v is not None and k != "set_timestamp_micros"
            )
            for item in items_list
        }
        data.append(row_dict)
    print(f"    → Extracted {len(data)} records")
    print(f"    ⏱️ Step time: {time.time() - start:.3f}s\n")

    # Step 2: Convert to DataFrame
    start = time.time()
    print("[2/4] Converting extracted data to DataFrame...")
    result_df = pd.DataFrame(data)
    print(f"    → DataFrame shape: {result_df.shape}")
    print(f"    ⏱️ Step time: {time.time() - start:.3f}s\n")

    # Step 3: Add prefix
    start = time.time()
    print("[3/4] Adding prefix to columns...")
    result_df = result_df.add_prefix(prefix)
    print(f"    → Prefixed columns: {list(result_df.columns)[:5]}...")
    print(f"    ⏱️ Step time: {time.time() - start:.3f}s\n")

    # Step 4: Merge with original DataFrame
    start = time.time()
    print("[4/4] Merging back with original DataFrame...")
    merged = pd.concat([df.reset_index(drop=True), result_df.reset_index(drop=True)], axis=1)
    print(f"    → Final merged shape: {merged.shape}")
    print(f"    ⏱️ Step time: {time.time() - start:.3f}s\n")

    total_time = time.time() - total_start
    print(f"=== ✅ Completed '{column_name}' in {total_time:.3f} seconds ===\n")

    return merged

In [10]:
df_2 = extract_nested_params(data_df,'event_params','ep_')
df_2


=== Processing column: 'event_params' with prefix: 'ep_' ===
[1/4] Extracting key/value pairs from nested structure...
    → Extracted 260864 records
    ⏱️ Step time: 3.457s

[2/4] Converting extracted data to DataFrame...
    → DataFrame shape: (260864, 28)
    ⏱️ Step time: 3.188s

[3/4] Adding prefix to columns...
    → Prefixed columns: ['ep_batch_page_id', 'ep_source', 'ep_page_location', 'ep_engaged_session_event', 'ep_campaign']...
    ⏱️ Step time: 0.128s

[4/4] Merging back with original DataFrame...
    → Final merged shape: (260864, 141)
    ⏱️ Step time: 2.368s

=== ✅ Completed 'event_params' in 9.141 seconds ===



,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,...,ep_click_url,ep_ignore_referrer,ep_click_classes,ep_click_id,ep_value,ep_link_text,ep_file_extension,ep_file_name,ep_link_url,ep_content
0,20241113,1731513971041603,first_visit,"[{'key': 'batch_page_id', 'value': {'string_va...",None,None,-2094340797,None,None,2091574202.1731513971,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20241113,1731513971041603,session_start,"[{'key': 'ga_session_id', 'value': {'string_va...",None,None,-2094340797,None,None,2091574202.1731513971,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20241113,1731513971041603,page_view,"[{'key': 'ga_session_number', 'value': {'strin...",None,None,-2094340797,None,None,2091574202.1731513971,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20241113,1731513976070304,user_session_info,"[{'key': 'batch_ordering_id', 'value': {'strin...",None,None,-2089312096,None,None,2091574202.1731513971,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20241113,1731513976070304,scroll,"[{'key': 'page_location', 'value': {'string_va...",None,None,-2089312096,None,None,2091574202.1731513971,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260859,20241113,1731552192524186,user_session_info,"[{'key': 'ga_session_id', 'value': {'string_va...",None,None,1767403418,None,None,1436230623.1728949201,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
260860,20241113,1731552192524186,scroll,"[{'key': 'batch_page_id', 'value': {'string_va...",None,None,1767403418,None,None,1436230623.1728949201,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
260861,20241113,1731552192524186,scroll,"[{'key': 'percent_scrolled', 'value': {'string...",None,None,1767403418,None,None,1436230623.1728949201,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
260862,20241113,1731552192524186,scroll,"[{'key': 'batch_ordering_id', 'value': {'strin...",None,None,1767403418,None,None,1436230623.1728949201,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df_2 = extract_nested_params(df_2,'user_properties','up_')
df_2


=== Processing column: 'user_properties' with prefix: 'up_' ===
[1/4] Extracting key/value pairs from nested structure...
    → Extracted 260864 records
    ⏱️ Step time: 0.752s

[2/4] Converting extracted data to DataFrame...
    → DataFrame shape: (260864, 2)
    ⏱️ Step time: 0.241s

[3/4] Adding prefix to columns...
    → Prefixed columns: ['up_user_session_id', 'up_user_client_id']...
    ⏱️ Step time: 0.028s

[4/4] Merging back with original DataFrame...
    → Final merged shape: (260864, 143)
    ⏱️ Step time: 4.688s

=== ✅ Completed 'user_properties' in 5.709 seconds ===



,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,...,ep_click_classes,ep_click_id,ep_value,ep_link_text,ep_file_extension,ep_file_name,ep_link_url,ep_content,up_user_session_id,up_user_client_id
0,20241113,1731513971041603,first_visit,"[{'key': 'batch_page_id', 'value': {'string_va...",None,None,-2094340797,None,None,2091574202.1731513971,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20241113,1731513971041603,session_start,"[{'key': 'ga_session_id', 'value': {'string_va...",None,None,-2094340797,None,None,2091574202.1731513971,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20241113,1731513971041603,page_view,"[{'key': 'ga_session_number', 'value': {'strin...",None,None,-2094340797,None,None,2091574202.1731513971,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20241113,1731513976070304,user_session_info,"[{'key': 'batch_ordering_id', 'value': {'strin...",None,None,-2089312096,None,None,2091574202.1731513971,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,_1731513970,_2091574202.1731513971
4,20241113,1731513976070304,scroll,"[{'key': 'page_location', 'value': {'string_va...",None,None,-2089312096,None,None,2091574202.1731513971,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,_1731513970,_2091574202.1731513971
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260859,20241113,1731552192524186,user_session_info,"[{'key': 'ga_session_id', 'value': {'string_va...",None,None,1767403418,None,None,1436230623.1728949201,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201
260860,20241113,1731552192524186,scroll,"[{'key': 'batch_page_id', 'value': {'string_va...",None,None,1767403418,None,None,1436230623.1728949201,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201
260861,20241113,1731552192524186,scroll,"[{'key': 'percent_scrolled', 'value': {'string...",None,None,1767403418,None,None,1436230623.1728949201,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201
260862,20241113,1731552192524186,scroll,"[{'key': 'batch_ordering_id', 'value': {'strin...",None,None,1767403418,None,None,1436230623.1728949201,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201


In [ ]:
params_data = [
        {
            item["key"]: next(v for v in item["value"].values() if v is not None )
            for item in params_list
        }
        for params_list in data_df["event_params"]
    ]
param_df = pd.DataFrame(params_data).add_prefix("ep_")

In [50]:
param_df

,ep_batch_page_id,ep_source,ep_page_location,ep_engaged_session_event,ep_campaign,ep_batch_ordering_id,ep_page_title,ep_medium,ep_session_engaged,ep_ga_session_id,...,ep_click_url,ep_ignore_referrer,ep_click_classes,ep_click_id,ep_value,ep_link_text,ep_file_extension,ep_file_name,ep_link_url,ep_content
0,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,4pc,0,1731513970,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,4pc,1,1731513970,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,4pc,0,1731513970,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1731513969906,NaN,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,NaN,2,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,1731513970,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1731513969906,NaN,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,NaN,2,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,1731513970,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260859,1731552185721,NaN,https://www.shopko.com/eye-care/?utm_source=em...,1.0,NaN,2,Eye Exam Center Locations Near You,NaN,0,1731552186,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
260860,1731552185721,NaN,https://www.shopko.com/eye-care/?utm_source=em...,1.0,NaN,2,Eye Exam Center Locations Near You,NaN,0,1731552186,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
260861,1731552185721,NaN,https://www.shopko.com/eye-care/?utm_source=em...,1.0,NaN,2,Eye Exam Center Locations Near You,NaN,0,1731552186,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
260862,1731552185721,NaN,https://www.shopko.com/eye-care/?utm_source=em...,1.0,NaN,2,Eye Exam Center Locations Near You,NaN,0,1731552186,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
user_props_data = [
        {
            prop["key"]: next(
                v
                for k, v in prop["value"].items()
                if v is not None and k != "set_timestamp_micros"
            )
            for prop in props_list
        }
        for props_list in data_df["user_properties"]
    ]
user_props_df = pd.DataFrame(user_props_data).add_prefix("user_prop_")

In [15]:
user_props_df

,user_prop_user_session_id,user_prop_user_client_id
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,_1731513970,_2091574202.1731513971
4,_1731513970,_2091574202.1731513971
...,...,...
260859,_1731552186,_1436230623.1728949201
260860,_1731552186,_1436230623.1728949201
260861,_1731552186,_1436230623.1728949201
260862,_1731552186,_1436230623.1728949201


In [16]:
event_date_dt = pd.to_datetime(data_df["event_date"], format="%Y%m%d")
data_df['year']= event_date_dt.dt.year
data_df['month'] = event_date_dt.dt.month

In [17]:
final_df = pd.concat([data_df, param_df, user_props_df], axis=1)
final_df.drop(columns=['user_properties', 'event_params'], inplace=True)

In [18]:
end_time = time.time()
end_time_str = datetime.datetime.now().strftime("%H:%M:%S")
print(end_time_str)
total_time = end_time - start_time
print(f"Total processing time without data loading and exporting: {total_time:.2f} seconds")

11:00:58
Total processing time without data loading and exporting: 182.96 seconds


In [19]:
final_df.to_csv("../CSV_output/events_normalized.csv", index=False)

In [20]:
total_end_time = time.time()
total_end_time_str = datetime.datetime.now().strftime("%H:%M:%S")
total_time = total_end_time - start_time_full
print(f"Total processing time: {total_time:.2f} seconds")

Total processing time: 230.75 seconds


In [21]:
print(76.5/60)

1.275


In [22]:
import pandas as pd

In [23]:
data = pd.read_csv("../CSV_output/events_normalized.csv", low_memory=False)


In [24]:
cols = data.columns.tolist()

ep_cols = [c for c in cols if c.startswith("ep_")]

before = []
after = []
passed_event_name = False

for c in cols:
    if c == "event_name":
        passed_event_name = True
        before.append(c)
    elif c.startswith("ep_"):
        continue
    else:
        if not passed_event_name:
            before.append(c)
        else:
            after.append(c)

new_order = before + ep_cols + after

data = data[new_order]

In [25]:
data.to_parquet("../parquet_output/events_normalized.parquet", engine="pyarrow", index=False)


In [26]:
data = pd.read_parquet("../parquet_output/events_normalized.parquet")

In [27]:
data

,event_date,event_timestamp,event_name,ep_batch_page_id,ep_source,ep_page_location,ep_engaged_session_event,ep_campaign,ep_batch_ordering_id,ep_page_title,...,session_traffic_source_last_click_google_ads_campaign_campaign_id,session_traffic_source_last_click_google_ads_campaign_campaign_name,session_traffic_source_last_click_google_ads_campaign_ad_group_id,session_traffic_source_last_click_google_ads_campaign_ad_group_name,user_ltv_revenue,user_ltv_currency,year,month,user_prop_user_session_id,user_prop_user_client_id
0,20241113,1731513971041603,first_visit,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,...,None,None,None,None,NaN,None,2024,11,None,None
1,20241113,1731513971041603,session_start,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,...,None,None,None,None,NaN,None,2024,11,None,None
2,20241113,1731513971041603,page_view,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,...,None,None,None,None,NaN,None,2024,11,None,None
3,20241113,1731513976070304,user_session_info,1731513969906,None,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,None,2,Eye Care Center in Crystal Lake - Shopko Optical,...,None,None,None,None,NaN,None,2024,11,_1731513970,_2091574202.1731513971
4,20241113,1731513976070304,scroll,1731513969906,None,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,None,2,Eye Care Center in Crystal Lake - Shopko Optical,...,None,None,None,None,NaN,None,2024,11,_1731513970,_2091574202.1731513971
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260859,20241113,1731552192524186,user_session_info,1731552185721,None,https://www.shopko.com/eye-care/?utm_source=em...,1.0,None,2,Eye Exam Center Locations Near You,...,None,None,None,None,NaN,None,2024,11,_1731552186,_1436230623.1728949201
260860,20241113,1731552192524186,scroll,1731552185721,None,https://www.shopko.com/eye-care/?utm_source=em...,1.0,None,2,Eye Exam Center Locations Near You,...,None,None,None,None,NaN,None,2024,11,_1731552186,_1436230623.1728949201
260861,20241113,1731552192524186,scroll,1731552185721,None,https://www.shopko.com/eye-care/?utm_source=em...,1.0,None,2,Eye Exam Center Locations Near You,...,None,None,None,None,NaN,None,2024,11,_1731552186,_1436230623.1728949201
260862,20241113,1731552192524186,scroll,1731552185721,None,https://www.shopko.com/eye-care/?utm_source=em...,1.0,None,2,Eye Exam Center Locations Near You,...,None,None,None,None,NaN,None,2024,11,_1731552186,_1436230623.1728949201
